In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local").appName("SparkFunctionPart2").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 07:54:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/19 07:54:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/19 07:54:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [1]:
from pyspark.sql.types import Row

# class Row(builtins.tuple)
#  |  Row(*args: Optional[str], **kwargs: Optional[Any]) -> 'Row'
#  |  
#  |  A row in :class:`DataFrame`.
#  |  The fields in it can be accessed:
#  |  
#  |  * like attributes (``row.key``)
#  |  * like dictionary values (``row[key]``)

# ``key in row`` will search through row keys.

row1 = Row(name="alice", age=11)

print(type(row1))

# access row properties
print("accessed using row1['colName']", row1['name'], row1['age'])

print('accessed using row1.colName', row1.name, row1.age)

print('accessed using position: ', row1[0], row1[1])

print('check if a key exists in a row object')

print("'name' in row1", 'name' in row1)

print("'wrongName' in row1", 'wrongName' in row1)



<class 'pyspark.sql.types.Row'>
accessed using row1['colName'] alice 11
accessed using row1.colName alice 11
accessed using position:  alice 11
check if a key exists in a row object
'name' in row1 True
'wrongName' in row1 False


In [5]:
#  Row also can be used to create another Row like class, then it
#  |  could be used to create Row objects, such as

Person = Row("name", "age")
p1 = Person("Shishir", 24)
p2 = Person("Rahul", 23)

print('type(Person)', type(Person))
print('type(p1)', type(p1))
print(type('Shishir'))

spark.createDataFrame(data = [p1,p2]).show()

print(p1.age, p1.name)

type(Person) <class 'pyspark.sql.types.Row'>
type(p1) <class 'pyspark.sql.types.Row'>
<class 'str'>


+-------+---+
|   name|age|
+-------+---+
|Shishir| 24|
|  Rahul| 23|
+-------+---+

24 Shishir


In [6]:
data = [Row(name="Shishir", age=24), \
        Row(name="Rahul", age=23), \
        Row(name="wafa", age=28)]

df = spark.createDataFrame(data)
df.show()
df.printSchema()

+-------+---+
|   name|age|
+-------+---+
|Shishir| 24|
|  Rahul| 23|
|   wafa| 28|
+-------+---+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [7]:
# Row() class can take named/unnamed args -- but you need to be consistent with one pattern either named or unnamed
data = [Row("Shishir", 24), \
        Row("Rahul", 23), \
        Row("wafa", 28)]

df = spark.createDataFrame(data, schema = ["name", "age"])
df.show()
df.printSchema()

+-------+---+
|   name|age|
+-------+---+
|Shishir| 24|
|  Rahul| 23|
|   wafa| 28|
+-------+---+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [8]:
r1 = Row("alice", 11)
r2 = Row(name="alice", age=11)
r3 = Row(11, "alice")

print(r1, r2, r3)

# I can access fields only using positions here
print(r1[0], r1[1])

# Here I can access via position as well as using keys

print(r2[0], r2[1])
print(r2['name'], r2.age)

# comparing 2 rows - considers positions and the actual values
print('r1 == r2', r1 == r2)

print('r1 == r3', r1 == r3)

<Row('alice', 11)> Row(name='alice', age=11) <Row(11, 'alice')>
alice 11
alice 11
alice 11
r1 == r2 True
r1 == r3 False


In [5]:
# nested rows

from pyspark.sql.types import Row
data = [Row(name="alice", prop=Row(age=11, gender="Female")), \
        Row(name="bob", prop=Row(age=12, gender="Male"))]

# 2 properties name: str, prop: struct age: long gender
# Here I have not specified the schema explicitly. Spark is auto inferring the datatype
# notice: how nested row is inferred as StructType
df = spark.createDataFrame(data)
df.printSchema()
df.show()

root
 |-- name: string (nullable = true)
 |-- prop: struct (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- gender: string (nullable = true)

+-----+------------+
| name|        prop|
+-----+------------+
|alice|{11, Female}|
|  bob|  {12, Male}|
+-----+------------+



In [38]:
# Column class

# Pyspark Column class in module pyspark.sql.column represents a single column in a DataFrame:

# Column class provides several functions to work with DataFrame to manipulate the column values, evaluate boolean expressions to filter rows etc

# One of the simplest ways to create a Column class object is by using lit() function

from pyspark.sql.functions import lit
from pyspark.sql.types import *

propertiesType = StructType([StructField("gender", StringType()), StructField("hairColor", StringType())])

schema = StructType().add("age", IntegerType()) \
                    .add("name", StringType()) \
                    .add("properties", propertiesType) 

df = spark.createDataFrame([(2, "Alice", ("male", "brown")), (5, "Bob", ("female", "blue"))], schema = schema)

df.printSchema()
# lit('abcd') -> Column
# It creates a Column expression that represents: “For every row, return the constant value 'abcd'.”
# This column expression / column object is not tied to any existing DataFrame column.
col1 = lit('abcd').alias("stringVal")
print(type(col1))


# Question: although "stringVal" column doesn't exist in DataFrame df, we were able to select that column WHy?

# select() does not only accept existing columns.
# It accepts expressions of type Column.

# So valid inputs to select are:
# Existing columns: col("age")
# Derived columns: col("age") + 1
# Constant columns: lit(5)
# Any expression returning a Column

df.select(lit('abcd').alias("stringVal"), # constant expression
          col('age') + 10).show()  # derived expression
# spark will evaluate the expressoin for each row.


# In plain Python, this should not work:

# df.select(...)
# df = spark.createDataFrame(...)

# Because Python executes top-to-bottom.
# If df truly didn’t exist at that moment, you would get:

# NameError: name 'df' is not defined
# Python never allows using a variable before it exists.

# df.select(...)   # builds a logical plan (lazy)
# .show()          # triggers execution
# But Python still needs df to exist to even build that logical plan.

# Meaning:
# Python: must know what df is immediately
# Spark: waits to run the job until an action like show() is called


# Mental model
# There are two kinds of things you pass to select:

# Column references

# col("age")     → look up existing column


# Column expressions

# lit(5)         → constant per row
# col("age") + 1 → derived per row
# when(...)      → conditional per row


# So:

# select(...) = "build new columns from expressions, not just pick existing ones"


# That’s why Spark lets you “select” columns that never existed before — you’re actually creating them on the fly.




# select a column out of a DataFrame - 4 ways

# using col() function -- col(colName)
# using df.colName
# using df['colName']
# using positions df[0]
df.select(col('name')).show()
df.select(df.name).show()
df.select(df['name']).show()
df.select(df[0]).show() 

# selecting structType fields
df.select(df.properties.gender).show()
df.select(col('properties.gender')).show()  # col('colName') -- you can only pass column name which exists in the dataFrame to the col() function
df.select(df['properties.gender']).show()

root
 |-- age: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- gender: string (nullable = true)
 |    |-- hairColor: string (nullable = true)

<class 'pyspark.sql.column.Column'>
+---------+----------+
|stringVal|(age + 10)|
+---------+----------+
|     abcd|        12|
|     abcd|        15|
+---------+----------+

+-----+
| name|
+-----+
|Alice|
|  Bob|
+-----+

+-----+
| name|
+-----+
|Alice|
|  Bob|
+-----+

+-----+
| name|
+-----+
|Alice|
|  Bob|
+-----+

+---+
|age|
+---+
|  2|
|  5|
+---+

+-----------------+
|properties.gender|
+-----------------+
|             male|
|           female|
+-----------------+

+------+
|gender|
+------+
|  male|
|female|
+------+

+------+
|gender|
+------+
|  male|
|female|
+------+



In [36]:
# creating columns using expression

df.age   # Column<'age'>
df.age + 1 # Column<'(age + 1)'>
(df.age + 1).alias("agePlusOne") # Column<'(age + 1) AS agePlusOne'>

# select agePlusOne column -- using df.select(ColumnName)
df.select((df.age + 1).alias("agePlusOne")).collect()

[Row(agePlusOne=3), Row(agePlusOne=6)]

In [43]:
df = spark.createDataFrame([('abcedfg', {"key": "value"})], ["l", "d"])

df.select(df.l, df.d.key).collect()

[Row(l='abcedfg', d[key]='value')]

In [59]:
df = spark.createDataFrame([('Tom', 80), ('Alice', None), ('Bob', 100)], ["name", "height"])

# by default, sorts in ascending fashion. You can either use asc()/desc() functions on column expressions or pass ascending parameter which is by default True

print('sort by height in asc order')
df.sort(col("height"), ascending = False).show()

print('sort by name in asc order')
df.sort(df.name.asc()).show()

print('sort by name in desc order using desc() function')
df.orderBy(df.name.desc()).show()

print('sort by name in desc order using ascending paramenter')
df.orderBy(df.name, ascending = False).show()


# both df.sort() and df.orderBy() actually does the same thing. sort() and orderBy() functions are aliases of each other.
# Under the hood, They both map to the same Spark SQL operation. They both produce same logical and physical plans.
# So why do both exist?
# Mainly for readability and familiarity:

# orderBy → closer to SQL: ORDER BY

# sort → more Pythonic / DataFrame-style


# df.height -- Column<'height'> gives u an instance of Column class
# Column class has bunch of method like slice(), asc(), desc(), asc_nulls_first(), asc_nulls_last(), desc_nulls_first(), desc_nulls_last()
# startswith(), endswith(), eqNullSafe(), like(), rlike(), ilike() etc

# select height, name from tbl order by height asc;
print('sort by height in asc order')
df.select(df.height, df.name).orderBy(df.height.asc()).show()

print('sort by name in desc order')
df.select(df.height, df.name).orderBy(df.name.desc()).show()

# There are some methods which are applied on top of DataFrames, whereas some are applied on top of Column instances. 

print('sort by height in asc order with null last')
df.select(df.height).orderBy(df.height.asc_nulls_last()).show()

print('sort by height in desc order with null first')
df.select(df.height).orderBy(df.height.desc_nulls_first()).show()

sort by height in asc order
+-----+------+
| name|height|
+-----+------+
|  Bob|   100|
|  Tom|    80|
|Alice|  NULL|
+-----+------+

sort by name in asc order
+-----+------+
| name|height|
+-----+------+
|Alice|  NULL|
|  Bob|   100|
|  Tom|    80|
+-----+------+

sort by name in desc order using desc() function
+-----+------+
| name|height|
+-----+------+
|  Tom|    80|
|  Bob|   100|
|Alice|  NULL|
+-----+------+

sort by name in desc order using ascending paramenter
+-----+------+
| name|height|
+-----+------+
|  Tom|    80|
|  Bob|   100|
|Alice|  NULL|
+-----+------+

sort by height in asc order
+------+-----+
|height| name|
+------+-----+
|  NULL|Alice|
|    80|  Tom|
|   100|  Bob|
+------+-----+

sort by name in desc order
+------+-----+
|height| name|
+------+-----+
|    80|  Tom|
|   100|  Bob|
|  NULL|Alice|
+------+-----+

sort by height in asc order with null last
+------+
|height|
+------+
|    80|
|   100|
|  NULL|
+------+

sort by height in desc order with null first


In [40]:
# slice(startPos, length) --> Column
df.select(slice(col("name"),1,2).alias("sliced")).collect()

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "slice(name, 1, 2)" due to data type mismatch: Parameter 1 requires the "ARRAY" type, however "name" has the type "STRING".;
'Project [slice(name#1086, 1, 2) AS sliced#1099]
+- LogicalRDD [name#1086, height#1087L], false


In [75]:
df.show()

# df.height -- Column instance
# on top of which we apply between(lowerBound, upperBound) 
# between() works in a way that it's inclusive of both lowerBound and upperBound
# i.e >= lowerBound and <= upperBound
df.filter(df.height.between(75,80)).collect()

+-----+------+
| name|height|
+-----+------+
|  Tom|    80|
|Alice|  NULL|
|  Bob|   100|
+-----+------+



[Row(name='Tom', height=80)]

In [80]:
# df.name -- returns Column instance on top of which .contains(str) method is applied
# use df.select() to select specific columns, but gives returns all Rows -- returns a DataFrame
# use df.filter() to filter specific rows, but returns all column -- also returns a DataFrame
df.filter(df.name.contains('o')).show()

# using both df.select().filter() together to choose specific columns and rows which satisfy a condition
df.select(df.name).filter(df.name.contains('o')).show()

+----+------+
|name|height|
+----+------+
| Tom|    80|
| Bob|   100|
+----+------+

+----+
|name|
+----+
| Tom|
| Bob|
+----+



In [90]:
from pyspark.sql.types import Row
df = spark.createDataFrame([Row(a=Row(b=1, c=2, d=3, e=Row(f=4, g=5, h=6)))])
df.show()
df.printSchema()

# dropFields drops fields in :class:`StructType` by name.
    # This is a no-op if the schema doesn't contain field name(s).
    
df.select(df.a.dropFields('b', 'e.f')).printSchema()

+--------------------+
|                   a|
+--------------------+
|{1, 2, 3, {4, 5, 6}}|
+--------------------+

root
 |-- a: struct (nullable = true)
 |    |-- b: long (nullable = true)
 |    |-- c: long (nullable = true)
 |    |-- d: long (nullable = true)
 |    |-- e: struct (nullable = true)
 |    |    |-- f: long (nullable = true)
 |    |    |-- g: long (nullable = true)
 |    |    |-- h: long (nullable = true)

root
 |-- update_fields(a, dropfield(), WithField(update_fields(update_fields(a, dropfield()).e, dropfield()))): struct (nullable = true)
 |    |-- c: long (nullable = true)
 |    |-- d: long (nullable = true)
 |    |-- e: struct (nullable = true)
 |    |    |-- g: long (nullable = true)
 |    |    |-- h: long (nullable = true)



In [115]:
df = spark.createDataFrame([Row(a = Row(b = 1, c = 2)), Row(a = Row(b = 3, c = 4))])
df.printSchema()
df.show()


# df.withColumn(colName: str, col: pyspark.sql.column.Column) -> DataFrame 
# returns a new DataFrame by adding a column or replacing the existing colun that has the same name

# withField(fieldName: str, col: 'Column') -> 'Column'
 # An expression that adds/replaces a field in :class:`StructType` by name. IF the fieldName exists, it replaces the field
df.withColumn('a', df.a.withField('b', lit(3))).show()

# you can select column in any of the 3 ways df.colname, df['colname'[ or col('colname')
# field 'd' doesn't exist, so it gets added
df.withColumn('a', df['a'].withField('d', lit(3))).show()

df.withColumn('a', col('a').withField('b', lit(3))).show()

# dropFields(*fieldNames: str) -> 'Column' method of pyspark.sql.column.Column instance : An expression that drops fields in :class:`StructType` by name.
    # This is a no-op if the schema doesn't contain field name(s).
    # ALso you cannot drop all fields
df.withColumn('a', df.a.dropFields('b')).show()

root
 |-- a: struct (nullable = true)
 |    |-- b: long (nullable = true)
 |    |-- c: long (nullable = true)

+------+
|     a|
+------+
|{1, 2}|
|{3, 4}|
+------+

+------+
|     a|
+------+
|{3, 2}|
|{3, 4}|
+------+

+---------+
|        a|
+---------+
|{1, 2, 3}|
|{3, 4, 3}|
+---------+

+------+
|     a|
+------+
|{3, 2}|
|{3, 4}|
+------+

+---+
|  a|
+---+
|{2}|
|{4}|
+---+



In [116]:
df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], ["age", "name"])
df.filter(df.name.endswith("ice")).collect()

[Row(age=2, name='Alice')]

In [137]:
df = spark.createDataFrame([
             Row(id=1, value='foo'),
             Row(id=2, value=None)])

df.show()

# df.id.eqNullSafe
 # Equality test that is safe for null values.
df.select(df.value == "foo", 
          df.value.eqNullSafe("foo"), 
          df.value.eqNullSafe(None)).show()

df2 = spark.createDataFrame([
            Row(value = 'bar'),
            Row(value = None)
    ])

# While joining use == comparison operator

# NULL == NULL - is false
df.join(df2, df.value == df2.value).show()

# using eqNullSafe() null == null is considered True
df.join(df2, df.value.eqNullSafe(df2.value)).show()

+---+-----+
| id|value|
+---+-----+
|  1|  foo|
|  2| NULL|
+---+-----+

+-------------+---------------+----------------+
|(value = foo)|(value <=> foo)|(value <=> NULL)|
+-------------+---------------+----------------+
|         true|           true|           false|
|         NULL|          false|            true|
+-------------+---------------+----------------+

+---+-----+-----+
| id|value|value|
+---+-----+-----+
+---+-----+-----+

+---+-----+-----+
| id|value|value|
+---+-----+-----+
|  2| NULL| NULL|
+---+-----+-----+



In [149]:
# df.height.isNotNull() -> Column
# df.height.isNull()
# True if the current expression is NOT null.

df = spark.createDataFrame([Row(name='Tom', height=80), Row(name='Alice', height=None)])

df.filter(df.height.isNotNull()).show()

# df.height.contains
df.filter(df.name.contains('i')).show()

df.filter(df.name.isin("Bob", "Tom")).show()

+----+------+
|name|height|
+----+------+
| Tom|    80|
+----+------+

+-----+------+
| name|height|
+-----+------+
|Alice|  NULL|
+-----+------+

+----+------+
|name|height|
+----+------+
| Tom|    80|
+----+------+



In [43]:
df = spark.range(3)
df.show()

from pyspark.sql.functions import *
# when(condition: pyspark.sql.column.Column, value: Any) -> pyspark.sql.column.Column
#     Evaluates a list of conditions and returns one of multiple possible result expressions.
#     If :func:`pyspark.sql.Column.otherwise` is not invoked, None is returned for unmatched
#     conditions.


# Parameters
#     ----------
#     condition : :class:`~pyspark.sql.Column`
#         a boolean :class:`~pyspark.sql.Column` expression.
#     value :
#         a literal value, or a :class:`~pyspark.sql.Column` expression.
    
df.select('id', when(df.id == 2, 4).otherwise(5).alias("age")).show()

df = spark.createDataFrame(data = [[1, "maheer", 'M', 2000], [1, "Asi", 'F', 2000], [1, "abcd", None, 2000]], schema = ["id", "name", "gender", "salary"])
df.show()


from pyspark.sql.functions import when, otherwise

# update the 'gender' column based on the values
# NOTE: while comparing values use == (comparison operator). = is (assignment operator)
df.withColumn('gender', when(df.gender == "M", "Male") \
                        .when(df.gender == "F", "Female") \
                        .otherwise("unknown")).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+

+---+---+
| id|age|
+---+---+
|  0|  5|
|  1|  5|
|  2|  4|
+---+---+

+---+------+------+------+
| id|  name|gender|salary|
+---+------+------+------+
|  1|maheer|     M|  2000|
|  1|   Asi|     F|  2000|
|  1|  abcd|  NULL|  2000|
+---+------+------+------+

+---+------+-------+------+
| id|  name| gender|salary|
+---+------+-------+------+
|  1|maheer|   Male|  2000|
|  1|   Asi| Female|  2000|
|  1|  abcd|unknown|  2000|
+---+------+-------+------+



In [88]:
# cast function

df = spark.createDataFrame(data = [[1,"maheer",2000], [2,"wafa",3000], [3,"asi",6000]], schema = ['id', 'name', 'salary'])

df.show()
df.printSchema()

# df.withColumn(colName, Column) gives you a new DataFrame. It doesn't modify the existing DataFrame.
# If the colName exists, it replaces the existing column with the new column expression, else adds it as a new column — but all this happens on the new DataFrame.
# Your original DataFrame state isn't altered.

df2 = df.withColumn('id', df.id.cast('int'))
df2.printSchema()


# df.select() also returns a new DataFrame - but only with selected set of columns. Performs transformations like casting etc on the fly. But this doesn't affect/alter our original DataFrame.
# Spark DataFrames are immutable by design.
df.select(df.id.cast("int")).printSchema()


# How to compare 2 columns across different dataframes if they are equal?
print(df.id == df2.id) # It doesn't compare data., just builds a column expression.

# To compare the data between 2 columns across different DataFrame, you need to join them and then perform comparison inside the joined DataFrame

df1 = spark.createDataFrame(data = [(1, "Shishir"), (2, "Rahul"), (3, "maheer")], schema = ["id", "name"])
df2 = spark.createDataFrame(data = [(1, "Shishir"), (2, "Patil"), (3, "Maheer")], schema = ["id", "name"])

# I want to compare name column across df1 and df2

# by default, spark performs inner join
joined = df1.alias("a").join(df2.alias("b"), df1.id == df2.id)
joined.show()

# ofcourse, this is case sensitive match
joined.select(df1.id, df1.name, df2.name, (df1.name == df2.name).alias("areNamesEqual")).show()

print('case insensitive matching')
joined.select(df1.id, df1.name, df2.name, (upper(df1.name) == upper(df2.name)).alias("areNamesEqual")).show()



# like() function

# say you want to filter rows where name contains 'a'
df1.select(df1.name, df1.name.like("%a%")).show()

# I can also filter to extract only those rows where name like "%a%"

# This will give you all the columns from the filtered DataFrame
df1.filter(df1.name.like("%a%")).show()

# You can select columns if you want on top of this - 
# My understanding: first filter -- this will filter rows , but provides you all columns. Then apply select on top of it to select the desired columns.
# But for selecting columns I can probably only use col() function and not the df.colName syntax because I haven't stored the filtered DataFrame. Can you verify my understanding?

# This is not entirely True, we can use both syntax for selecting columns here. Both will work bcoz spark resolves the column references at the query planning time, not by python variable chaining logic.
# df1.name is just syntactic sugar for col("name") bound to df1. But in chained calls, Spark still resolves it correctly.
df1.filter(df1.name.like("%a%")).select(col("name")).show()

df1.filter(df1.name.like("%a%")).select(df1.name).show()
# so both syntaxes for selecting columns work after filtering the DataFrame



# filter() first and then select()
# or select() first and then filter()
# Are both these 2 valid and are there any performance implications?

# Both are valid and both produce the same result (assuming your filter condition only uses columns that are still present).

# Example
# df.filter(df.name.like("%a%")).select("name")
# df.select("name").filter(col("name").like("%a%"))

# Both return only rows where name contains "a" and only the name column.

# Performance: practically the same
# Spark’s optimizer (Catalyst) will rewrite your query.
# It will:
# Push filter as early as possible
# Drop unused columns as early as possible
# So internally, Spark turns both into an optimized plan like:
# Scan → Filter → Project

# Regardless of the order you wrote.
# This is called:
# Predicate pushdown
# Column pruning

# When order does matter
# If you drop a column before filtering on it:
# df.select("name").filter(col("age") > 30)   # ❌ age no longer exists

# If filter uses derived columns:
# df.select((df.age + 10).alias("new_age")).filter(col("new_age") > 40)  # valid
# df.filter(df.age + 10 > 40).select("age")                              # also valid

# Both are fine, but logic must match. Choose the order that makes your code most readable — Spark will optimize it anyway.

# But Remember if you are perform select() first and then filter() --> filter column must be present in the select(). Otherwise it becomes invalid since you dropped the column from the select

+---+------+------+
| id|  name|salary|
+---+------+------+
|  1|maheer|  2000|
|  2|  wafa|  3000|
|  3|   asi|  6000|
+---+------+------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)

root
 |-- id: integer (nullable = true)

Column<'(id = id)'>
+---+-------+---+-------+
| id|   name| id|   name|
+---+-------+---+-------+
|  1|Shishir|  1|Shishir|
|  2|  Rahul|  2|  Patil|
|  3| maheer|  3| Maheer|
+---+-------+---+-------+

+---+-------+-------+-------------+
| id|   name|   name|areNamesEqual|
+---+-------+-------+-------------+
|  1|Shishir|Shishir|         true|
|  2|  Rahul|  Patil|        false|
|  3| maheer| Maheer|        false|
+---+-------+-------+-------------+

case insensitive matching
+---+-------+-------+-------------+
| id|   name|   name|areNamesEqual|
+---+-------+-------+-------------+
|  1|

In [42]:
from pyspark.sql.functions import filter, month, to_date

# month(col: 'ColumnOrName') -> pyspark.sql.column.Column
#     Extract the month of a given date/timestamp as integer.
    
    
# filter(col: 'ColumnOrName', function) -> pyspark.sql.column.Column
 # Parameters
 #    ----------
 #    col : :class:`~pyspark.sql.Column` or str
 #        name of column or expression
 #    f : function
 #        A function that returns the Boolean expression.

    
# to_date(col: ColumnOrName, format: Optional) ->  pyspark.sql.column.Column 
# ate(col: 'ColumnOrName', format: Optional[str] = None) -> pyspark.sql.column.Column
#     Converts a :class:`~pyspark.sql.Column` into :class:`pyspark.sql.types.DateType`
#     using the optionally specified format. Specify formats according to `datetime pattern`_.
#     By default, it follows casting rules to :class:`pyspark.sql.types.DateType` if the format
#     is omitted. Equivalent to ``col.cast("date")``.

df = spark.createDataFrame(
             [(1, ["2018-09-20",  "2019-02-03", "2019-07-01", "2020-06-01"])],
            ("key", "values")
    )

df.show(truncate=False)

def afterSecondQuarter(date_column):
    return month(to_date(date_column)) > 6

df.select(filter(df.values, afterSecondQuarter).alias("datesAfterSecondQtr")).show(truncate=False)

df.select(filter("values", afterSecondQuarter).alias("datesAfterSecondQtr")).show(truncate=False)


# df.filter() and df.where()
# filter(condition: 'ColumnOrName') -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
# df.where(condition: 'ColumnOrName') -->  'DataFrame'

# They both return you a filtered DataFrame based on the given Column Expression

# df.where() is an alias of df.filter() and they work in exaclty same manner
df = spark.createDataFrame(data = [(1, "Shishir", 3000), (2, "Rahul", 5000), (3, "maheer", 6000)], schema = ["id", "name", "salary"])
df.show()

print('filter where id >= 2 and salary > 5500')
df.filter((df.id >= 2) & (df.salary >= 5500)).show()
df.select(df.id, df.id >= 2).show()


### NOTE: there are 2 variations filter function. The first variation df.filter() returns a DataFrame, whereas 2nd variation filter(ColumnOrName, functionName) --> returns a Column instance 

    # 1. df.filter(condition: 'ColumnOrName') on DataFrame instane which returns filtered DataFrame
    # 2. filter(col: ColumnOrName, functionName) -> pyspark.sql.column.Column from pyspark.sql.functions 
    # 3. df.filter() and df.where() they both work exactly in same manner.
    
    
    # All these method below returns a DataFrame. Refer to doc of DataFrame Class to see all the available methods available on top of DataFrame instance. 
    # Similarly we have Row
    
    # spark.createDataFrame()
    # spark.range()
    # df.filter()
    # df.where()
    # df.select()
    # df.withColumn()
    # df.withColumnRenamed()
    
    
#     Important classes that we''ve seen so far :
        # DataFrame(module pyspark.sql.dataframe:), Column(module pyspark.sql.column), Row(in module pyspark.sql.types)  
        # SparkSession(module pyspark.sql.session), StructType(module pyspark.sql.types), StructField(module pyspark.sql.types)
        # DataFrameReader(created via spark.read) -- Key methods: csv, json, parquet, orc, format, option, options, schema, load
        # DataFrameWriter(Created via df.write) Key methods: mode, format, partitionBy, save, saveAsTable, parquet, csv, json
        # GroupedData(Returned by df.groupBy() Key methods: agg, count, sum, avg, max, min)
        # WindowSpec: Used for window functions: Key methods: partitionBy, orderBy, rowsBetween, rangeBetween)

        # RDD / Low-level Layer
        # RDD Lower-level distributed collection
        # Key methods: map, flatMap, filter, reduceByKey collect, take, count

        # SparkContext: Underlying engine handle
        # sc = spark.sparkContext
        # Key methods:parallelize, textFile, broadcast, accumulator
        
#        Streaming Layer
#        DataStreamReader: spark.readStream
#        DataStreamWriter: df.writeStream

#         Types & Schema Layer
#         14) StructType, StructField
#             Schema definition

#         15) DataType and subclasses
#             StringType, IntegerType, etc.

#         Utility / Execution
#         16) Broadcast

#         Used for broadcast joins
#         17) Accumulator
        # Shared counters across executors

#     help(DataFrame)
    
    
    
    

+---+------------------------------------------------+
|key|values                                          |
+---+------------------------------------------------+
|1  |[2018-09-20, 2019-02-03, 2019-07-01, 2020-06-01]|
+---+------------------------------------------------+

+------------------------+
|datesAfterSecondQtr     |
+------------------------+
|[2018-09-20, 2019-07-01]|
+------------------------+

+------------------------+
|datesAfterSecondQtr     |
+------------------------+
|[2018-09-20, 2019-07-01]|
+------------------------+

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|Shishir|  3000|
|  2|  Rahul|  5000|
|  3| maheer|  6000|
+---+-------+------+

filter where id >= 2 and salary > 5500
+---+------+------+
| id|  name|salary|
+---+------+------+
|  3|maheer|  6000|
+---+------+------+

+---+---------+
| id|(id >= 2)|
+---+---------+
|  1|    false|
|  2|     true|
|  3|     true|
+---+---------+



In [73]:
df = spark.createDataFrame(
      [(14, "Tom"), (23, "Alice"), (23, "Alice")], ["age", "name"])

# distinct() -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
#     Returns a new :class:`DataFrame` containing the distinct rows in this :class:`DataFrame`.

df.distinct().show()
# help(df.distinct)

+---+-----+
|age| name|
+---+-----+
| 14|  Tom|
| 23|Alice|
+---+-----+



In [81]:
from pyspark.sql import Row

# dropDuplicates(subset: Optional[List[str]] = None) -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
#     Return a new :class:`DataFrame` with duplicate rows removed,
df = spark.createDataFrame([
    Row(name='Alice', age=5, height=80),
    Row(name='Alice', age=5, height=80),
    Row(name='Alice', age=10, height=80)
 ])

# By default, it considers all the columns to figure out duplication. You can also pass a list of column name on the basis of which you want to perform deduplication

# Ofcourse, dropDuplicates() returns you a new DataFrame. Your original DataFrame is not modified.
print('unique rows')
df.dropDuplicates().show()

print('unique names')
df.dropDuplicates(["name"]).show()

print('unique name, height combos')
df.dropDuplicates(["name", "height"]).show()

print('unique name, age combos')
df.dropDuplicates(["name", "age"]).show()


# df.distinct()
# - Returns unique rows based on all columns
# - Takes no arguments
# - Equivalent to SQL: SELECT DISTINCT *

# df.dropDuplicates()
# - By default: removes duplicate rows using all columns
# - Can take a subset of columns: dropDuplicates(["col1", "col2"])
# - Keeps the first occurrence (based on current row order)

# So:

# If you want:
# - Unique rows across all columns → use df.distinct()
# - Unique rows based on specific column(s) → use df.dropDuplicates(["col"])
# - Unique combinations of columns → use df.dropDuplicates(["col1", "col2"])

# df.distinct()   ==   df.dropDuplicates()
# when no columns are passed.

# df.distinct() works on all columns and does not take any arguments.
# df.dropDuplicates() by default also works on all columns, but it can accept a list of column names.
# So if you want a unique list of values for a column or unique combinations across selected columns, df.dropDuplicates() is the right choice.

unique rows
+-----+---+------+
| name|age|height|
+-----+---+------+
|Alice|  5|    80|
|Alice| 10|    80|
+-----+---+------+

unique names
+-----+---+------+
| name|age|height|
+-----+---+------+
|Alice|  5|    80|
+-----+---+------+

unique name, height combos
+-----+---+------+
| name|age|height|
+-----+---+------+
|Alice|  5|    80|
+-----+---+------+

unique name, age combos
+-----+---+------+
| name|age|height|
+-----+---+------+
|Alice| 10|    80|
|Alice|  5|    80|
+-----+---+------+



In [88]:
data = [(1, 'maheer', 'M', 2000, 'IT'), (2, 'wafa', 'M', 4000, 'HR'), (3, 'aso', 'F', 3000, 'Payroll'), (4, 'raj', 'M', 3000, 'HR')]
schema = ['id', 'name', 'gender', 'salary', 'dept']

df = spark.createDataFrame(data, schema)
df.show()

# We can use sort() or orderBy() function of Pyspark DataFrame to sort DataFrame by ascending or descending order based on single or multiple columns
# By default sorting wil happen in ascending order. We can explicitly mention asc or desc order using asc(), desc() function
df.sort('dept', 'salary').show()
df.sort(df.dept.desc(), df.salary.desc()).show()
df.orderBy(df.dept.desc(), df.salary.asc()).show()
df.orderBy(df.dept, df.salary).show()

+---+------+------+------+-------+
| id|  name|gender|salary|   dept|
+---+------+------+------+-------+
|  1|maheer|     M|  2000|     IT|
|  2|  wafa|     M|  4000|     HR|
|  3|   aso|     F|  3000|Payroll|
|  4|   raj|     M|  3000|     HR|
+---+------+------+------+-------+

+---+------+------+------+-------+
| id|  name|gender|salary|   dept|
+---+------+------+------+-------+
|  4|   raj|     M|  3000|     HR|
|  2|  wafa|     M|  4000|     HR|
|  1|maheer|     M|  2000|     IT|
|  3|   aso|     F|  3000|Payroll|
+---+------+------+------+-------+

+---+------+------+------+-------+
| id|  name|gender|salary|   dept|
+---+------+------+------+-------+
|  3|   aso|     F|  3000|Payroll|
|  1|maheer|     M|  2000|     IT|
|  2|  wafa|     M|  4000|     HR|
|  4|   raj|     M|  3000|     HR|
+---+------+------+------+-------+

+---+------+------+------+-------+
| id|  name|gender|salary|   dept|
+---+------+------+------+-------+
|  3|   aso|     F|  3000|Payroll|
|  1|maheer|     

In [107]:
data = [(1, 'maheer', 'M', 2000, 'IT'), (2, 'wafa', 'M', 4000, 'HR'), (3, 'aso', 'F', 3000, 'Payroll'), (4, 'raj', 'M', 3000, 'HR')]
schema = ['id', 'name', 'gender', 'salary', 'dept']

df = spark.createDataFrame(data, schema)
df2 = spark.createDataFrame(data, schema)
df.show()

# df.union(df2) -> pyspark.sql.dataframe.DataFrame
# and df.unionAll(df2) -> pyspark.sql.dataframe.DataFrame

print('df.union(df2).show()')
df.union(df2).show()

print('df.unionAll(df2).show()')
df.unionAll(df2).show()

# NOTE: notice how both union and unionAll() are yielding same output. In pyspark, union and unionAll() are aliases of each other. You have to use distinct() to remove duplicates.
# In sql, union removes duplicates whereas unionAll() retains duplicates. BUT in pyspark, the behaviour is little different. Both union() and unionAll() retains all duplicates.

# union() and unionAll() are used to merge 2 or more DataFrames of the same schema or structure.
# By same structure I mean, they can have different column names but the no of columns should be same
# or Column names are same but their order is different -- all these means they have same structure and thus union operation works (although you might see incorrect results)

# NOTE: The method resolves columns by position (not by name). Just like in sql. This is the key for union operation.
# To remove duplicates use df.distinct() function
# :func:`unionAll` is an alias to :func:`union`

# union() vs join()
# union merges 2 or more DataFrames in a vertical fashion (one below another). Only prerequisite is they should have same structure (basically same no of columns, column names and order could be different - union operation wont fail, but you wil see incorrect results)
# join merges 2 or more DataFrames in a horizontal fashion (side by side) on the basis of a key(s)


# --------

print('Example 1: Combining two DataFrames with the same schema')
   
df1 = spark.createDataFrame([(1, 'A'), (2, 'B')], ['id', 'value'])
df2 = spark.createDataFrame([(3, 'C'), (4, 'D')], ['id', 'value'])
df3 = df1.union(df2)
df3.show()


print('Example 2: Combining two DataFrames with different schemas') # -  by same schema it means - exactly same set of columns and following the same order.
print('Here the columns are same, but the order is different (essentially same structure). Union operation doesn''t fail as such, but produces corrupt results (columns swapped)')
    
from pyspark.sql.functions import lit
df1 = spark.createDataFrame([("Alice", 1), ("Bob", 2)], ["name", "id"])
df2 = spark.createDataFrame([(3, "Charlie"), (4, "Dave")], ["id", "name"])
df1 = df1.withColumn("age", lit(30))
df2 = df2.withColumn("age", lit(40))
df3 = df1.union(df2)
df3.show()


print('Example 3: Combining two DataFrames with mismatched columns -  here the column names are different')
# Union works
    
df1 = spark.createDataFrame([(1, 2)], ["A", "B"])
df2 = spark.createDataFrame([(3, 4)], ["C", "D"])
df3 = df1.union(df2)
df3.show()


print('Example 4: Combining duplicate rows from two different DataFrames - same columns and same order')
 # union retains duplicates - have to use distinct() or dropDuplicates() to remove the duplicates

df1 = spark.createDataFrame([(1, 'A'), (2, 'B'), (3, 'C')], ['id', 'value'])
df2 = spark.createDataFrame([(3, 'C'), (4, 'D')], ['id', 'value'])
df3 = df1.union(df2).sort("id")
df3.show()


+---+------+------+------+-------+
| id|  name|gender|salary|   dept|
+---+------+------+------+-------+
|  1|maheer|     M|  2000|     IT|
|  2|  wafa|     M|  4000|     HR|
|  3|   aso|     F|  3000|Payroll|
|  4|   raj|     M|  3000|     HR|
+---+------+------+------+-------+

df.union(df2).show()
+---+------+------+------+-------+
| id|  name|gender|salary|   dept|
+---+------+------+------+-------+
|  1|maheer|     M|  2000|     IT|
|  2|  wafa|     M|  4000|     HR|
|  3|   aso|     F|  3000|Payroll|
|  4|   raj|     M|  3000|     HR|
|  1|maheer|     M|  2000|     IT|
|  2|  wafa|     M|  4000|     HR|
|  3|   aso|     F|  3000|Payroll|
|  4|   raj|     M|  3000|     HR|
+---+------+------+------+-------+

df.unionAll(df2).show()
+---+------+------+------+-------+
| id|  name|gender|salary|   dept|
+---+------+------+------+-------+
|  1|maheer|     M|  2000|     IT|
|  2|  wafa|     M|  4000|     HR|
|  3|   aso|     F|  3000|Payroll|
|  4|   raj|     M|  3000|     HR|
|  1|mah

In [6]:
# df.groupBy() --> :class:`GroupedData`
 # Groups the :class:`DataFrame` using the specified columns,
 #    so we can run aggregation on them. See :class:`GroupedData`
 #    for all the available aggregate functions.
    

df = spark.createDataFrame([(2, "Alice", 180), (2, "Bob", 165), (2, "Bob", 174), (5, "Bob", 120)], schema=["age", "name", "height"])
df.show()

df.groupBy()  # returns GroupedData object- grouping expression [] since we have not specified any columns for grouping, value: list of columns and their datatypes
# GroupedData[grouping expressions: [], value: [age: bigint, name: string ... 1 more field], type: GroupBy]
df.groupBy().max() # returns DataFrame[max(age): bigint, max(height): bigint]
df.groupBy().max().show()  # returns max(age), max(height) across entire DataFrame
# Notice: how same aggregation function is used for across all the fact columns. If you want to perform different kinds of aggregation, we will use agg({"age": "max", "height": "min"})

df.groupBy("name").max().show() # returns max(age), max(height) across each name

df.groupBy("name").agg({"age": "max", "height": "min"}).withColumnRenamed('max(age)', 'max_age').show()  # groupBy name and return max(age) and min(height) for each group
# also renamed the column using withColumnRenamed(oldColumnName, newColumnName) -> DataFrame

df.groupBy("name").count().sort("count").show()
df.groupBy("name").agg({"*": "count"}).show()


df.groupBy("name").min("age").withColumnRenamed("min(age)", "min_age").show()

df.groupBy("name").count().show()

df.groupBy("name").sum("age", "height").show()


# agg({"colname": "agg_function"}) -- 
print("agg function is usefull when you want to apply different kinds of aggregation at the same time.")
df.groupBy("name").agg({"age":"min", "height":"max"}).show()

print("count of rows and sum(height) across entire DataFrmae")
# agg({"colName": "aggFunction"}) 
# count is not used on any specific column -- doing that will give us count(colName) will give you count of not null rows in that column
df.groupBy().agg({"*": "count", "height": "sum"}).show()

# Other way we could write this agg() function is 

from pyspark.sql.functions import count, sum
# The count() function in PySpark has different behaviors depending on how it is used. 
# It can be an action to return the total number of rows in a DataFrame, or an aggregation function to count items in a group or column. 
df.groupBy().agg(count("*").alias("count"), sum("height").alias("sumHeight")).show()



# Pivot is used to rotate data in one column into multiple columns.
# Its an aggregation where one of the grouping column values will be converted into individual column


# pivot(self, pivot_col: str, values: Optional[List[ForwardRef('LiteralType')]] = None) -> 'GroupedData'
#  |      Pivots a column of the current :class:`DataFrame` and perform the specified aggregation.
#  |      There are two versions of the pivot function: one that requires the caller
#  |      to specify the list of distinct values to pivot on, and one that does not.
#  |      The latter is more concise but less efficient,
#  |      because Spark needs to first compute the list of distinct values internally.


from pyspark.sql import Row
df1 = spark.createDataFrame([
        Row(course="dotNET", year=2012, earnings=10000),
        Row(course="Java", year=2012, earnings=20000),
        Row(course="dotNET", year=2012, earnings=5000),
        Row(course="dotNET", year=2013, earnings=48000),
        Row(course="Java", year=2013, earnings=30000),
    ])

df1.show()

#  Compute the sum of earnings for each year by course with each course as a separate column

# This is the the normal grouping
df1.groupBy("year", "course").sum("earnings").show()

# now say I want the course values as separate column -- meaning I need to pivot on the course column
df1.groupBy("year").pivot("course").sum("earnings").show()

# second version of writing pivot() function -- is supply the unique list of course column values 
# This is more efficient bcoz in the latter one spark needs to compute the list of distinct values internally.
df1.groupBy("year").pivot("course", ["Java", "dotNET"]).sum("earnings").show()



df2 = spark.createDataFrame([
         Row(training="expert", sales=Row(course="dotNET", year=2012, earnings=10000)),
         Row(training="junior", sales=Row(course="Java", year=2012, earnings=20000)),
         Row(training="expert", sales=Row(course="dotNET", year=2012, earnings=5000)),
         Row(training="junior", sales=Row(course="dotNET", year=2013, earnings=48000)),
         Row(training="expert", sales=Row(course="Java", year=2013, earnings=30000)),
    ])  # doctest: +SKIP

df2.show() 

# compute sum(earnings) for each year by course with each course as a separate column

# groupBy("year") and pivot("course") -- now since these columns are part of sales struct
# df2.groupBy("sales.year") -> GroupedData object
# df2.groupBy("sales.year").pivot("sales.course") # -> GroupedData object
df2.groupBy("sales.year").pivot("sales.course").sum("sales.earnings").show()

    
    
# GroupedData class has provides several aggregate functions
    # avg(), min(), max(), sum(), count() -- built in functions
    # agg({"colname": "agg_function"}) -- agg function is usefull when you want to apply different kinds of aggregation at the same time.
    # pivot(colName: str , listOfUniqueValues: Optional) -> GroupedData
    # apply(), applyInPandas() --  PENDING
    
    
    
# help(GroupedData)


+---+-----+------+
|age| name|height|
+---+-----+------+
|  2|Alice|   180|
|  2|  Bob|   165|
|  2|  Bob|   174|
|  5|  Bob|   120|
+---+-----+------+

+--------+-----------+
|max(age)|max(height)|
+--------+-----------+
|       5|        180|
+--------+-----------+

+-----+--------+-----------+
| name|max(age)|max(height)|
+-----+--------+-----------+
|  Bob|       5|        174|
|Alice|       2|        180|
+-----+--------+-----------+

+-----+-------+-----------+
| name|max_age|min(height)|
+-----+-------+-----------+
|  Bob|      5|        120|
|Alice|      2|        180|
+-----+-------+-----------+

+-----+-----+
| name|count|
+-----+-----+
|Alice|    1|
|  Bob|    3|
+-----+-----+

+-----+--------+
| name|count(1)|
+-----+--------+
|  Bob|       3|
|Alice|       1|
+-----+--------+

+-----+-------+
| name|min_age|
+-----+-------+
|  Bob|      2|
|Alice|      2|
+-----+-------+

+-----+-----+
| name|count|
+-----+-----+
|  Bob|    3|
|Alice|    1|
+-----+-----+

+-----+--------+-

In [7]:
# DataFrame 1: Columns in order 'name', 'id'
df1 = spark.createDataFrame([("Alice", 1)], ["name", "id"])
df1.printSchema()

# DataFrame 2: Columns in order 'id', 'name' (different order)
df2 = spark.createDataFrame([(2, "Bob")], ["id", "name"])
df2.printSchema()

# The unionByName() function in PySpark combines two or more DataFrames by matching columns based on their names rather than their positions. 
# This makes it a robust solution for merging datasets where column order may differ or schemas may evolve. 

# Key Features and Usage
# Column Alignment: Unlike the standard union() function, which requires identical column order, unionByName() automatically aligns columns with the same name.

# Handling Missing Columns: By default, if one DataFrame has a column that the other doesn't, it results in an error. However, setting the allowMissingColumns=True parameter will automatically add the missing column to the other DataFrame and fill the corresponding rows with null values.

# Data Types: Columns with the same name must have compatible data types; otherwise, Spark will throw an error. 

# df.union(df2) and df.unionAll(df2) -> DataFrame 
# By default union() and unionAll() functions resolves columns based on their positions and not on their name. So even if the column names are different/follows different order, but the structure(number of columns) is same in both DataFrames, union() will not fail.
# In case of unionByName() -- columns are resolved based on their names while the union operation takes place. So, if the columns with same names are not present (their order could be different - that's fine, spark will resolve it because they have same names)
# spark will throw an error. To fix this, pass `allowMissingColumns` is True. Spark will add the missing columns in the corresponsding DataFrames and union operation will succeed.

# unionByName method performs a union operation on both input DataFrames, resolving columns by
# name (rather than position). When `allowMissingColumns` is True, missing columns will be filled with null.
df1.union(df2).show()

# despite df1 and df2 having different column orders -- union operation is performed correctly, because we used unionByName() function
df1.unionByName(df2).show()

print("Union with missing columns and setting `allowMissingColumns=True`.")
    
df1 = spark.createDataFrame([[1, 2, 3]], ["col0", "col1", "col2"])
df2 = spark.createDataFrame([[4, 5, 6]], ["col1", "col2", "col3"])
df1.unionByName(df2, allowMissingColumns=True).show()

# NOTE: There are several functions which behave differently depending on how they are being used. For instance , count() can be used as an aggregate function or count() in the DataFrame class - df.count() 
# similary filter() -- df.filter() function in the DataFrame class and filter() from pyspark.sql.functions import filter()

root
 |-- name: string (nullable = true)
 |-- id: long (nullable = true)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)

+-----+---+
| name| id|
+-----+---+
|Alice|  1|
|    2|Bob|
+-----+---+

+-----+---+
| name| id|
+-----+---+
|Alice|  1|
|  Bob|  2|
+-----+---+

Union with missing columns and setting `allowMissingColumns=True`.
+----+----+----+----+
|col0|col1|col2|col3|
+----+----+----+----+
|   1|   2|   3|NULL|
|NULL|   4|   5|   6|
+----+----+----+----+



In [41]:
df = spark.createDataFrame([(2, "Alice"), (5, "Bob")]).toDF("age", "name")
df3 = spark.createDataFrame([Row(age=2, name="Alice"), Row(age=5, name="Bob")])

from pyspark.sql.functions import desc, col

# df.join()
  # Parameters
  #   ----------
  #   other : :class:`DataFrame`
  #       Right side of the join

  #   on : str, list or :class:`Column`, optional
  #       a string for the join column name, a list of column names,
  #       a join expression (Column), or a list of Columns.
  #       If `on` is a string or a list of strings indicating the name of the join column(s),
  #       the column(s) must exist on both sides, and this performs an equi-join.
    
  #   how : str, optional
  #       default ``inner``.
 
  #    Must be one of: ``inner``, ``cross``, ``outer``,
  #       ``full``, ``fullouter``, ``full_outer``, ``left``, ``leftouter``, ``left_outer``,
  #       ``right``, ``rightouter``, ``right_outer``, ``semi``, ``leftsemi``, ``left_semi``,
  #       ``anti``, ``leftanti`` and ``left_anti``.
    

# different methods using which you can sort the final DataFrame. 
df.join(df3, (df.age == df3.age) & (df.name == df3.name), 'outer').select(df.age, df3.name).sort(desc("age")).show()

df.join(df3, (df.age == df3.age) & (df.name == df3.name), 'outer').select(df.age, df3.name).sort(col("age").desc()).show()

df.join(df3, (df.age == df3.age) & (df.name == df3.name), 'outer').select(df.age, df3.name).sort("age", ascending = False).show()

# commmon join types that we know: inner, left, right, and full outer join

# # some other joins that are avaialble here are : self, left-anti, left-semi
# left-semi join is similar to inner join, but only gets columns from the left dataFrame for matching rows
# left-anti join is opposite to left-semi, gets you non matching rows from the left dataFrame

emp_df = spark.createDataFrame(data = [(1, "shishir", 2000, 1), (2, "rahul", 3000, 2), (3, "Ajay", 1500, 4)], schema = ["id", "name", "salary", "dept_id"])

dept_df = spark.createDataFrame(data = [(1, "IT"), (2, "HR"), (3,"Marketing"), (4, "Finance")], schema = ["id", "dept_name"])

print('inner join: default join')
emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'inner').show()

print('full_outer join')
emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'full_outer').show()

print('left join')
emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'left').show()

print('right join')
emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'right').show()

print('left_semi join')
emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'left_semi').show()

print('left_anti join')
emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'left_anti').show()

print('self join')

emp_df = spark.createDataFrame(data = [(1, "shishir", 0), (2, "rahul", 1), (3, "Ajay", 2)], schema = ["id", "name", "mgr_id"])
emp_df.show()
emp_df.alias("tbl1").join(emp_df.alias("tbl2"), col("tbl1.mgr_id") == col("tbl2.id"), 'left').show()


# Even though you aliased the DataFrames, Spark still sees both sides coming from the same logical plan, so column references like mgr_id and id can’t be resolved safely unless you fully qualify them.
tbl1 = emp_df.alias("tbl1")
tbl2 = emp_df.alias("tbl2")
# tbl1.join(tbl2, tbl1.mgr_id == tbl2.id, 'left').show()    # Throws ambigous column exception

tbl1.join(tbl2, col("tbl1.mgr_id") == col("tbl2.id"), 'left').show() 

# inner join => matching rows
# left join => all rows from left table (matched + unmatched) both
# right join => all rows from right table (matched + unmatched) both
# full outer join => all rows from left and right tables (matched + left unmatched + right unmatched)
# left-semi join => only matching rows and left side column from the inner join
# left-anti join => only unmatched rows from the left dataFrame
# self join => joins data with the same DataFrame

+---+-----+
|age| name|
+---+-----+
|  5|  Bob|
|  2|Alice|
+---+-----+

+---+-----+
|age| name|
+---+-----+
|  5|  Bob|
|  2|Alice|
+---+-----+

+---+-----+
|age| name|
+---+-----+
|  5|  Bob|
|  2|Alice|
+---+-----+

inner join: default join
+---+-------+------+-------+---+---------+
| id|   name|salary|dept_id| id|dept_name|
+---+-------+------+-------+---+---------+
|  1|shishir|  2000|      1|  1|       IT|
|  2|  rahul|  3000|      2|  2|       HR|
|  3|   Ajay|  1500|      4|  4|  Finance|
+---+-------+------+-------+---+---------+

full_outer join
+----+-------+------+-------+---+---------+
|  id|   name|salary|dept_id| id|dept_name|
+----+-------+------+-------+---+---------+
|   1|shishir|  2000|      1|  1|       IT|
|   2|  rahul|  3000|      2|  2|       HR|
|NULL|   NULL|  NULL|   NULL|  3|Marketing|
|   3|   Ajay|  1500|      4|  4|  Finance|
+----+-------+------+-------+---+---------+

left join
+---+-------+------+-------+---+---------+
| id|   name|salary|dept_id| id|

In [25]:
data = [("IT", 4, 3), \
       ("HR", 2, 3), \
       ("Finanace", 5, 3)]

schema = ["dept", "male", "female"]

df = spark.createDataFrame(data, schema)
df.show()

# unpivot - converts columns into rows using stack() function
# unpivoting — it turns a wide table into a long table.

# stack(2, 'Male', male, 'Female', female) as (gender, count)
# args:
    # 2: no of coloums to unpivot
    # 'male': value that you want to get from the unpivot col
    # male: col that you want to unpivot
    # 'female': value that you want to get from the unpivot col
    # female: col that you want to unpivot

    
# expr(str: str) -> pyspark.sql.column.Column
    # Parses the expression string into the column that it represents
    # expr() takes python/pyspark code as string and returns a column

from pyspark.sql.functions import expr, stack
unpivot_df = df.select("dept", expr("stack(2, 'Male', male, 'Female', female) as (gender, count)"))
unpivot_df.show()

# another way of writing this - using selectExpr
unpivot_df = df.selectExpr("dept", "stack(2, 'male', male, 'female', female) as (gender, count)")
unpivot_df.sort("dept", desc("count")).show()

+--------+----+------+
|    dept|male|female|
+--------+----+------+
|      IT|   4|     3|
|      HR|   2|     3|
|Finanace|   5|     3|
+--------+----+------+

+--------+------+-----+
|    dept|gender|count|
+--------+------+-----+
|      IT|  Male|    4|
|      IT|Female|    3|
|      HR|  Male|    2|
|      HR|Female|    3|
|Finanace|  Male|    5|
|Finanace|Female|    3|
+--------+------+-----+

+--------+------+-----+
|    dept|gender|count|
+--------+------+-----+
|Finanace|  male|    5|
|Finanace|female|    3|
|      HR|female|    3|
|      HR|  male|    2|
|      IT|  male|    4|
|      IT|female|    3|
+--------+------+-----+



In [62]:
# df = spark.createDataFrame([("Alice", ), ("Bob", )], ["name"]) 
# df1 = spark.createDataFrame(["Alice", "Bob"], ["name"])  X Invalid

data = [Row(name = "alice"), Row(name = "bob")]

# data = [("alice"), ("bob")]  X data must be list of Row objects or list of tuples. Each tuple or Row objects represents a row in DataFrame.
# ("alice") -> is not a tuple
# ("alice", ) -> This is a valid tuple
data = [("alice", ), ("bob", )]
df = spark.createDataFrame(data, ["name"])
df.show()
df.printSchema()

# df.select() vs df.selectExpr() 

# df.select() expects column objects whereas df.selectExpr expects sql like strings. They both return DataFrames with selected columns.
# The difference is in the type of arguments passed

# expr() is a bridge between SQL-style expressions and the DataFrame API.
# It lets you write SQL syntax, but use it inside normal DataFrame operations.

# What expr() does?
# Takes a SQL expression as a string
# Converts it into a Column object
# So it can be used anywhere a Column is expected

# Why it exists

# Because:

# selectExpr() → only accepts SQL strings

# select() → only accepts Column objects

# expr() lets you mix SQL-style logic into select, withColumn, filter, etc.

# Example

df.select(df.name, expr("length(name) as length")).show()


# All three below are equivalent:

# Hybrid using expr()
df.select(df.name, expr("length(name) as length")).show()

# SQL Style
df.selectExpr("name", "length(name) as length").show()

# DataFrame API style
from pyspark.sql.functions import length
df.select(df.name, length(df.name)).show()



# selectExpr() only expects sql like strings. You cannot pass Column objects
# df.selectExpr(df.name, "length(name)").show() X Invalid 


+-----+
| name|
+-----+
|alice|
|  bob|
+-----+

root
 |-- name: string (nullable = true)

+-----+------+
| name|length|
+-----+------+
|alice|     5|
|  bob|     3|
+-----+------+

+-----+------+
| name|length|
+-----+------+
|alice|     5|
|  bob|     3|
+-----+------+

+-----+------+
| name|length|
+-----+------+
|alice|     5|
|  bob|     3|
+-----+------+

+-----+------------+
| name|length(name)|
+-----+------------+
|alice|           5|
|  bob|           3|
+-----+------------+

+-----+------------+
| name|length(name)|
+-----+------------+
|alice|           5|
|  bob|           3|
+-----+------------+



In [78]:
# fill() and fillna() functions

# help(df.fillna)
# df.fillna() method is available on pyspark DataFrame object.
# which Replaces null values, alias for ``na.fill()`` and returns you a new DataFrame

# These functions are used to replace null values in your DataFrame

data = [(1, "Shishir", "Male", 200.50, "IT", True), \
       (1, None, "Male", 200.74, None, False), \
       (None, "Asi", "Female", None, "HR", None), \
       (1, "Raj", None, 200.80, "IT", None)]

schema = ["id", "name", "gender", "salary", "dept", "trustworthy"]

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

# say we want to replace all the string columns null values with 'unknown' -- we can use fill() or fillna() function
# The type of value fillna() accepts: int, float, string, bool or dict 
# Depending on the type of value passed to the fillna function, it will replace nulls in only those columns that has the same DataType.
# For instance, if you pass a number to fillna function, in the entire DataFrame what all columns have numeric values (float,int,double,long etc) - their null values will be replaced  with the value passed to fillna method
# If you pass string value, fillna function will replace nulls in all the string columns. Rest of the columns remains untouched.
# Assume you have 5 string columns and you want to fill nulls in only 3 of them and with different values. You can pass a dict {"colName: "value"}
# which will tell spark to replace nulls in each column with different set of values.


print("replaces nulls for all the string columns with 'unknown'")
df.fillna("unknown").show()

print("replace all nulls in the numeric columns with 0")
df.fillna(0).show()

print("replace all nulls in the boolean columns with False")
df.fillna(False).show()

print("replaces nulls in the name column with 'unnamed' and dept column with 'unknown'")
print('notice: gender which is a string column stays untouched')
df.fillna({"name": "unnamed", "dept": "unknown"}).show()

print("replace nulls with 'unknown' in name & gender column")
# "unknown' is the value with which you want to replace nulls
# 2nd parameter is the list of columns on which you want to replace nulls. If the list is not passed, spark will replace nulls in all the columns depending on the Datatype of the value passed
# since we've passed a string "unknown", spark will replace nulls in all the string columns across the entire DataFrame.

df.fillna("unknown", ["name", "gender"]).show()

# NOTE: df.na.fill() is an alias of df.fillna function -- both work in exactly the same way

+----+-------+------+------+----+-----------+
|  id|   name|gender|salary|dept|trustworthy|
+----+-------+------+------+----+-----------+
|   1|Shishir|  Male| 200.5|  IT|       true|
|   1|   NULL|  Male|200.74|NULL|      false|
|NULL|    Asi|Female|  NULL|  HR|       NULL|
|   1|    Raj|  NULL| 200.8|  IT|       NULL|
+----+-------+------+------+----+-----------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- dept: string (nullable = true)
 |-- trustworthy: boolean (nullable = true)

replaces nulls for all the string columns with 'unknown'
+----+-------+-------+------+-------+-----------+
|  id|   name| gender|salary|   dept|trustworthy|
+----+-------+-------+------+-------+-----------+
|   1|Shishir|   Male| 200.5|     IT|       true|
|   1|unknown|   Male|200.74|unknown|      false|
|NULL|    Asi| Female|  NULL|     HR|       NULL|
|   1|    Raj|unknown| 200.8|     IT|       

In [115]:
df = spark.range(100)

# say you have a huge dataset, and you want to get a subset of this Dataset for testing or for developement purpose. 
# We can use df.sample() to generate a sample of a Dataset

# sample method on Pyspark DataFrame object
# Returns a sampled subset of this :class:`DataFrame`.

# Parameter:
# fraction: specifies the percentage of data that you want sample() to return. Fraction is not accurate. You might get more or less data as well
# seed : specify seed value to make sure that every time you the same sample.

df1 = df.sample(fraction=0.1, seed=123)
df2= df.sample(fraction=0.1, seed=123)

df1.show()
df1.count() # Notice: how you get different no of rows in the sample each time. A random sample is generated which contains around 10% of data.

# assume you want to generate the same sample each time, for this puprose you can pass a seed. 
# Seed value makes sure that you get same random sample for each execution. 
# seed=123 --> the sample generated by this seed value will always be same. Different seeds will generate differnt samples

df2.show()



+---+
| id|
+---+
| 36|
| 37|
| 41|
| 43|
| 56|
| 66|
| 69|
| 75|
| 83|
+---+

+---+
| id|
+---+
| 36|
| 37|
| 41|
| 43|
| 56|
| 66|
| 69|
| 75|
| 83|
+---+



In [ ]:
# collect

# Returns all the records as a list of :class:`Row` to the driver node.
# Use collect() function cautiously, because with large DataFrames -- it might produce OOM errors
# It is an action and thus it does not return any DataFrame. It collects all the rows in your DataFrame as Row objects in a list inside your driver node.


# Spark stores DataFrames in a distributed way across worker nodes using partitions.

# Each worker node holds only a portion of the total data.

# The Driver does not store the full dataset by default — it only coordinates execution.

# When collect() is called, all partitions from all workers are sent to the Driver and combined into a single list of rows.

# After collect(), the entire dataset exists in the Driver’s memory.

# Once we have the list, we can loop over it and apply any logic that we want

DriverNode

WN1  WN2.   WN3   WN4

# In a Spark cluster, you have:

# One Driver Node – runs your application logic and coordinates work

# Multiple Worker Nodes – actually store data partitions and run tasks

# When you create a DataFrame with, say, 100 rows:

# The DataFrame is split into partitions

# These partitions are distributed across worker nodes

# For example:

# WN1 → rows 1–25

# WN2 → rows 26–50

# WN3 → rows 51–75

# WN4 → rows 76–100
# (Exact split depends on partitioning, not row count)

# So yes:

# Data is stored and processed in a distributed manner — not all rows live on a single node.




In [119]:
# transform() avialable on pyspark DataFrame object 

# It's used to chain custom transformation and this function returns a new DataFrame after applying the specified tranformations

data = [("Shishir", 2000), ("Preet", 2000), ("Rahul", 2000)] 

# create a DataFrame
df = spark.createDataFrame(data, schema = ["name", "salary"])

from pyspark.sql.functions import upper

# defined 2 custom functions which takes DataFrame as a parameter and returns a DataFrame
def convertToUpperCase(df):
    return df.withColumn('name', upper(df.name))

def doubleTheSalary(df):
    return df.withColumn('salary', df.salary * 2)

# using transform() we chain multiple transformations.
# promotes code reusability and keeps code clean and consise
df.transform(convertToUpperCase).transform(doubleTheSalary).show()

+-------+------+
|   name|salary|
+-------+------+
|SHISHIR|  4000|
|  PREET|  4000|
|  RAHUL|  4000|
+-------+------+



In [134]:
from pyspark.sql.functions import transform

# There are 2 varaints of transform function. 

# 1st : df.transform() which is avaialble on DataFrame instance object  

    # This transform function is applied on a DataFrame and returns a new DataFrame

# 2nd: transform() from pyspark.sql.functions module
    # This transform function is applied on array type column to transform the array values and returns array type column

# This transform function can only be applied on arraytype column 

    # transform(col: 'ColumnOrName', f: function) ->  pyspark.sql.column.Column
    
    # col - is an array type column
    
    # f - is a function which is applied to each element of the input array


data = [("Shishir", ["Java", "Python"]), ("Preet", ["AWS", "Golang"])] 

# create a DataFrame
df = spark.createDataFrame(data, schema = ["name", "skills"])
df.show()

# say, you want to convert skills to uppercase
# transform() from pyspark.sql.functions takes 2 args
# 1st: array column name or column object
# 2nd: lambda function - that will be applied on each element of the array
df.select(df.name, transform("skills", lambda x: upper(x)).alias("skills")).show()

+-------+--------------+
|   name|        skills|
+-------+--------------+
|Shishir|[Java, Python]|
|  Preet| [AWS, Golang]|
+-------+--------------+

+-------+--------------+
|   name|        skills|
+-------+--------------+
|Shishir|[JAVA, PYTHON]|
|  Preet| [AWS, GOLANG]|
+-------+--------------+



In [141]:
# createOrReplaceTempView()

# Biggest advantage of Spark is - you can work with SQL along with DataFrames. That means, if you are comfortable with SQL, you can create a temporary view on DataFrame by using createOrReplaceTempView function
# and use SQL to select and manipulate data. 

# This temp view is session scoped and is not available outside this spark session

# df.selectExpr() or expr() functions are some other functions which allows sql string to be passed

data = [(1, "Shishir", 2000), (2, "Rahul", 3000), (3, "Asi", 2000)]

df = spark.createDataFrame(data, schema = ['id', 'name', 'salary'])

# DataFrame API style
# seleting name & salary columns 
df.select("name", "salary").show()


# SQL Style
# you can do the same thing via sql 
    # 1. create a temp view on top of DataFrame
    # 2. Query the temp view using sql style
df.createOrReplaceTempView("employees")
result_df = spark.sql("select name, salary from employees")
result_df.show()

+-------+------+
|   name|salary|
+-------+------+
|Shishir|  2000|
|  Rahul|  3000|
|    Asi|  2000|
+-------+------+

+-------+------+
|   name|salary|
+-------+------+
|Shishir|  2000|
|  Rahul|  3000|
|    Asi|  2000|
+-------+------+



In [143]:
%sql
select name from employees;

SyntaxError: invalid syntax (2295838252.py, line 2)